In [11]:
#1. This cell reads the galaxy properties for all galaxies and load their precalculated moments for the dN/dV profile.

import csv 
import emcee
import numpy as np
from Multiphase_Wind_Fitting_function_for_Classy import *
import time
from multiprocessing import Pool
import numpy as np
import glob
from scipy import integrate, interpolate
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.colors as colors
from matplotlib import cm
from matplotlib.colors import ListedColormap
from scipy.integrate import ode
from scipy.integrate import solve_ivp
from scipy import optimize
from scipy.interpolate import interp1d
import matplotlib.font_manager
import h5py
from matplotlib.backends.backend_pdf import PdfPages
import os
import corner
import matplotlib.ticker as ticker
import pandas as pd
import re
from scipy.io import readsav

# Function to replace multiple tabs with a single tab
def preprocess_line(line):
    return re.sub(r'\t+', '\t', line)


# Read a range of columns using Excel-style column letters
def ReadExcel(FilePath):
    df = pd.read_excel(FilePath, usecols='A:O')


    objname2 = df['Objname']
    Mdot = df['Mdot_outflow (Msun/yr)']
    Pdot = df['log(Pdot_outflow) (dynes)']
    Edot = df['log(Edot_outflow) (erg/s)']

    Vout    = df['Vout (km/s)']
    FWHMout = df['FWHMout (km/s)']
    NHout   = df['log N_H out (cm^-2)']

    return objname2,Mdot,Pdot,Edot,Vout, FWHMout, NHout

# Read a range of columns using Excel-style column letters
def ReadExcel2(FilePath):
    df = pd.read_excel(FilePath, usecols='A:P')


    objname4 = df['objname']
    SFR4 = df['LogSFR']
  
    return objname4,SFR4

def clean_strings(string_list):
    cleaned_list = []
    for string in string_list:
        # Split the string by '^' sign and take the first part
        clean_string = string.split('^')[0]
        # Remove leading/trailing spaces and special characters
        clean_string = clean_string.strip().replace('--', '-').replace('+ ', '+').replace('- ', '-')
        cleaned_list.append(clean_string)
    return cleaned_list

#Split string like 0.5+0.2-0.1 to be value+-errors
#If the format does not match, it will return None, None, None.
def parse_errors(error_string):
    # Use a regular expression to find the value and errors, allowing for leading whitespace
    pattern = re.compile(r"\s*([+-]?[0-9]*\.?[0-9]+)([+-][0-9]*\.?[0-9]+)(-([0-9]*\.?[0-9]+))")
    

    match = pattern.match(error_string)
    if match:
        value = float(match.group(1))
        upper_error = float(match.group(2))
        lower_error = float(match.group(4))
        return value, upper_error, lower_error
    else:
        return None, None, None


#Read a space separated file
#Ncol is the total # of cols in the file.
#Nread is the total # of cols that you want to return.
#Line starting with # will be skipped
#Read a space separated file
#Ncol is the total # of cols in the file.
#Nread is the total # of cols that you want to return.
#Line starting with # will be skipped
#Example: objname0, objnameshort,zobj0 = ReadcolXX('./In0.0.txt', 3,3,delimiter = '\t')
def ReadcolXX(MasterFile, Ncol, Nread, delimiter):

    # Open the text file for reading
    with open(MasterFile, 'r') as f:

        #1. Create a csv reader object with space as the delimiter
        # skipinitialspace is to skip consecutive spaces
        #reader = csv.reader(f, delimiter=delimiter,skipinitialspace=True)
        reader = csv.reader((preprocess_line(line) for line in f), delimiter=delimiter, skipinitialspace=True)
    

        
        #2. Define a dictionary to store the different columns
        arr = {}
        for i in range(1,Nread+1):
            arr['arr'+str(i)] = [] #i.e., it will contain keys as arr1, arr2, arr3, etc
        #print(arr)

        # Loop through the rows and extract the columns
        for row in reader:
            #print(f"One line = {row}")
            
            if len(row) == Ncol:
                

                if row[0][0] == '#':
                    a = 1#placeholder
                    #print("This line is skipped since no data!")
                else:
                    for i in range(1,Nread+1):
                        arr['arr'+str(i)].append(row[i-1])


    return tuple(
        arr[f'arr{i}']
        for i in range(1, Nread + 1)
    )


#Read the measurements from CLASSY III paper and construct the observed moments array
def ReadObservations(MasterPATH):
    
    #MasterFile = os.path.join(MasterPATH, 'In1.1_CLASSYProcess2.txt') 
    #ID, objname, zobj = ReadcolXX(MasterFile, 20, 3,delimiter = '|')
    #print("objname = ",objname,len(objname))
    #print("zobj = ",zobj,len(zobj))

    #1.1 This file contains the outflow rates
    FilePath2 = os.path.join(MasterPATH, 'Out5.1_HLSP.xlsx')
    objname2, Mdot, Pdot, Edot, Vout, FWHMout, NHout = ReadExcel(FilePath2)
    

    #1.2 This file contains the R_50
    FilePath3 = os.path.join(MasterPATH, 'Out5.3_AncillaryParams.txt')    
    objname3, zobj3, r_50_arcsec, r_50_kpc, vcir_arr = ReadcolXX(FilePath3, 7,5,delimiter = '&')
    

    #1.3 This file contains SFR
    FilePath4 = os.path.join(MasterPATH, 'Out6.5_HLSP.xlsx')    
    objname4, SFR4 = ReadExcel2(FilePath4)

    #1.4 This file contains the measured v_peak and HWHM of the dN/dV profile
    FilePath5 = os.path.join('./Results/CLASSYIII/MeasureNH', 'Out10.0_NH_Profile_Info.txt')    
    objname5, mean_v_cloud, HWHM_cloud, NH_cloud, NH_int_left, NH_int_rigt = ReadcolXX(FilePath5, 6, 6,delimiter = ',')

    NMeaMethod = 2 # =1 will use the CLASSY III paper's Vout, FWHMout, and total NH
                   # =2 will use dN/dV profile's abs(Vout) and FWHMout, and the same NH as above (actually very similar as the ones measured from dN/dV)
                        #These values are measured in ./Calculate_NH_CLASSY_obs

    
    #print(f"objname5 = {objname5}")
    #print(f"mean_v_cloud = {mean_v_cloud}")
    #print(f"HWHM_cloud = {HWHM_cloud}")
    #print(f"NH_cloud = {NH_cloud}")


    
    #sanity check to ensure that the objectname are the same.
    objname2 = clean_strings(objname2) #remove extra - and ^{} for shorter names.
    objname3 = clean_strings(objname3)
    objname4 = clean_strings(objname4)
    objname5 = clean_strings(objname5)
    
    if np.array_equal(objname2, objname3) != True:
        STOP
    if np.array_equal(objname2, objname4) != True:
        STOP
    if np.array_equal(objname2, objname5) != True:
        STOP
        
    #print(f"objname3 = {objname3}")
    #print(f"objname5 = {objname5}")
    #raw
    
    #3. Define some params in my paper
    omega   = 1.0    #outflow solid angle. 
                                   #TBD: thinking about if I should use Omwind/(4*np.pi)*1
                                   #In CLASSY III, I used solid angle = 4pi, so the scaling is 1 here.
                                   #But for other Omwind != 4pi, you need to scale differently. e.g., 2 pi -> 0.5 as scaling.
    pi      = 3.1415
    miu     = 1.4        # average photon mass fraction
    cmTopc  = 1.0/pc     # 1cm to parsec
    kgTosun = 1.586e-23  # 1kg/s = A Msun/year
    gTosun  = kgTosun*1E-3
    mp      = 1.67373522381e-24

    #4. Calculate the moments
    #Outflows rates - used in model001 - 003
    #note I still output the actual Mdot, pdot, Edot to calculate the initial eta values 
    m1      = [0.0] * len(objname3)
    m2      = [0.0] * len(objname3)
    m3      = [0.0] * len(objname3)
    m1_err  = [0.0] * len(objname3)
    m2_err  = [0.0] * len(objname3)
    m3_err  = [0.0] * len(objname3)

    #v, sigma_v and N - used in model001_FitN
    #I used NX just to match the format of m1 - m3.
    N1      = [0.0] * len(objname3) #Vout or abs(mean_v_cloud) directly from dN/dv - FB model returns all positive v, which does not consider doppler effect
    N2      = [0.0] * len(objname3) #FWHMout or HWHM_cloud directly from dN/dv
    N3      = [0.0] * len(objname3) #Total NHout
    N1_err  = [0.0] * len(objname3) 
    N2_err  = [0.0] * len(objname3)
    N3_err  = [0.0] * len(objname3)

    Int_left= [0.0] * len(objname3) #Integration range left side for NH_tot (or NH_out here). Calculate from  ./Calculate_NH_CLASSY_obs
    Int_rigt= [0.0] * len(objname3)

    #galaxy properties
    vcir    = [0.0] * len(objname3)
    sfr     = [0.0] * len(objname3)   
    r50     = [0.0] * len(objname3) 

    #Errors - note used
    MdotErrUp     = [0.0] * len(objname3) 
    MdotErrDo     = [0.0] * len(objname3) 
    PdotErrUp     = [0.0] * len(objname3) 
    PdotErrDo     = [0.0] * len(objname3) 
    EdotErrUp     = [0.0] * len(objname3) 
    EdotErrDo     = [0.0] * len(objname3) 


    for obji in range(0,len(objname3)):
       
        

        #4.1 First try to split Mdot, Pdot, Edot into value and errors
        #If no valid outflow rates, it will return None.
        value1, upper_error1, lower_error1 = parse_errors(Mdot[obji])
        value2, upper_error2, lower_error2 = parse_errors(Pdot[obji])
        value3, upper_error3, lower_error3 = parse_errors(Edot[obji])
        value4, upper_error4, lower_error4 = parse_errors(vcir_arr[obji])
        value5, upper_error5, lower_error5 = parse_errors(Vout[obji])
        value6, upper_error6, lower_error6 = parse_errors(FWHMout[obji])
        value7, upper_error7, lower_error7 = parse_errors(NHout[obji])

        
        #4.2 Get the value and errors
        oneMdot    = [value1, upper_error1, lower_error1]        #Msun/yr
        onePdot    = [value2, upper_error2, lower_error2]        #dynes in log.
        oneEdot    = [value3, upper_error3, lower_error3]        #ergs/s in log. 10**(Edot[obji]) 
        onevcir    = [value4, upper_error4, lower_error4]        #km/s

        if NMeaMethod ==1:
            oneVout    = [value5, upper_error5, lower_error5]        #km/s
            oneFWHMout = [value6, upper_error6, lower_error6]        #km/s
            oneNHout   = [value7, upper_error7, lower_error7]        #cm^-2
        elif NMeaMethod == 2:

             #Binning of dN/dV is 40km/s, so error is fixed at 20km/s here
            oneVout    = [abs(float(mean_v_cloud[obji])), 20, 20]      #peak velocity measured from dN/dV profile. 
                                                                                          
            oneFWHMout = [abs(float(HWHM_cloud[obji])), 20, 20]        
                                                                                                                               
            oneNHout   = [abs(float(NH_cloud[obji])), upper_error7, lower_error7]        #cm^-2


            
            
            
            one_Int_left= NH_int_left[obji] #fill the integration ranges for NHout
            one_Int_rigt= NH_int_rigt[obji]            
        else:
            print(f"ERROR: you have not defined how to get the measurements of N1-N3! NMeaMethod = {NMeaMethod} not defined!")
            raw

        #print(f"oneVout = {oneVout}")
        #raw
        
        oner_50    = float(r_50_kpc[obji])                       #kpc
        r_50_cm    = 2.0*oner_50*1000.0/cmTopc

        #print(f"oneVout = {oneVout}")
        #print(f"oneFWHMout = {oneFWHMout}")
        #print(f"oneNHout = {oneNHout}")
        #raw
        
        #4.3 Calculate the moments
        #Note the input Mdot, Pdot, Edot are in different units
        #But the output m1, m2, m3 are all in cgs units.
        if oneMdot[0] != None:
            const      = omega*miu*mp*r_50_cm     #constant outside the integration: g*cm
            
            m1[obji]   = oneMdot[0]/gTosun  / const  #s^-1 cm^-1 = dN_H/dv *v
            m2[obji]   = 10**onePdot[0]/const        #= dN_H/dv *v^2
            m3[obji]   = 10**oneEdot[0]/0.5/const    #= dN_H/dv *v^3
            
            m1_err[obji]   = (oneMdot[1] + oneMdot[2])/2.0/gTosun  / const                           #(upper error +lower error)/2
            #m2_err[obji]   = (10**(onePdot[0] + onePdot[1]) - 10**(onePdot[0]-onePdot[2]))/2.0/const #(upper value -lower value) /2, note upper value = value +-error
            #m3_err[obji]   = (10**(oneEdot[0] + oneEdot[1]) - 10**(oneEdot[0]-oneEdot[2]))/2.0/0.5/const 
            m2_err[obji]   =m1_err[obji]/m1[obji] * m2[obji]  #the above errors propagration fails for large errorbars in log
            m3_err[obji]   =m1_err[obji]/m1[obji] * m3[obji]  #so I just assum Pdot and Edot has similar error percentage as Mdot.

            #The results from MWFF are in cgs unit and linear: v, sigma, NH = (20235283.643189132, 21940751.96652906, 4.2525979069319023e+21)
            N1[obji]   = np.log10(oneVout[0]*10**5)    #logVout in cm/s
            N2[obji]   = np.log10(oneFWHMout[0]*10**5) #logFWHMout in cm/s
            N3[obji]   = oneNHout[0]                   #logNH - already in log scale and in cm^-2
            Int_left[obji] = one_Int_left
            Int_rigt[obji] = one_Int_rigt

            #N1_err does not need to be converted to cgs since now it is in log scale.
            N1_err[obji] = (oneVout[1] + oneVout[2]) / (2.0 * oneVout[0] * np.log(10))   #error_log = error_linear/value_linear/ln(10)
            N2_err[obji] = (oneFWHMout[1] + oneFWHMout[2]) / (2.0 * oneFWHMout[0] * np.log(10))  
            N3_err[obji] = (oneNHout[1] + oneNHout[2])/2.0

            #raw#TBD change the unit and likely to log scale here
        
        else:
            m1[obji]       = None
            m2[obji]       = None
            m3[obji]       = None          
            m1_err[obji]   = None
            m2_err[obji]   = None
            m3_err[obji]   = None   

            N1[obji]   = None   
            N2[obji]   = None   
            N3[obji]   = None   
            Int_left[obji] = None
            Int_rigt[obji] = None
            
            N1_err[obji]   = None
            N2_err[obji]   = None   
            N3_err[obji]   = None   

        
        #4.4 Store other galaxy properties: I dont need errorbars here
        vcir[obji] = onevcir[0]
        sfr[obji]  = 10**SFR4[obji]
        r50[obji]  = oner_50

        #4.5 Store error bars for MPE dot. Note I overwrite the arrays for values.
        Mdot[obji]          = value1
        MdotErrUp[obji]     = upper_error1
        MdotErrDo[obji]     = lower_error1

        Pdot[obji]          = value2
        PdotErrUp[obji]     = upper_error2
        PdotErrDo[obji]     = lower_error2

        Edot[obji]          = value3
        EdotErrUp[obji]     = upper_error3
        EdotErrDo[obji]     = lower_error3

    
        #sanity checks
        ifCheck = 0
        if ifCheck == 1:     
            objicheck = 6
            if obji == objicheck:
                print(f"objname3 = {objname3[obji]}")
                print(f"Mdot, Pdot, Edot, r_50 = {oneMdot[0], onePdot[0], oneEdot[0],oner_50}")
                print(f"Mdot, Pdot, Edot, r_50 errup = {oneMdot[1], onePdot[1], oneEdot[1],oner_50}")
                print(f"Mdot, Pdot, Edot, r_50 errdo = {oneMdot[2], onePdot[2], oneEdot[2],oner_50}")
                print(f"m1, m2, m3, const = {m1[obji],m2[obji],m3[obji],const}")
                print(f"m1, m2, m3 err = {m1_err[obji],m2_err[obji],m3_err[obji]}")
                print(f"SFR, Vcir = {sfr[obji], vcir[obji]}")
    
                print(f"N1, N2, N3 value = {N1[obji],N2[obji],N3[obji]}")
                print(f"N1, N2, N3 err = {N1_err[obji],N2_err[obji],N3_err[obji]}")
                print(f"NH integration range = {Int_left[obji], Int_rigt[obji]}")
                raw


    
    return objname3, zobj3, m1, m1_err, m2, m2_err, m3, m3_err, N1, N1_err, N2, N2_err, N3, N3_err, Int_left, Int_rigt,r50, sfr, vcir, Mdot, Pdot, Edot, MdotErrUp, MdotErrDo, PdotErrUp, PdotErrDo, EdotErrUp, EdotErrDo



MasterPATH = './' #Note CLASSY files copied from '/Applications/Work-Desktop/CLASSY/Analysis/PlotSiII/'
objname, zobj, m1, m1_err, m2, m2_err, m3, m3_err,  \
    N1, N1_err, N2, N2_err, N3, N3_err, Int_left, Int_rigt, \
    r50, sfr, vcir,  \
    Mdot, Pdot, Edot, MdotErrUp, MdotErrDo, PdotErrUp, PdotErrDo, EdotErrUp, EdotErrDo = ReadObservations(MasterPATH)


print(f"objname = {objname}")
print(f"N1 = {N1}")
print(f"N2 = {N2}")
print(f"N3 = {N3}")

#    N1      = logVout in cm/s from dN/dv - FB model returns all positive v, which does not consider doppler effect
#    N2      = logHWHMout in cm/s directly from dN/dv
#    N3      = log of total NH in cm^-2


objname = ['J0021+0052', 'J0036-3333', 'J0055-0021', 'J0127-0619', 'J0144+0453', 'J0150+1308', 'J0337-0502', 'J0405-3648', 'J0808+3948', 'J0823+2806', 'J0926+4427', 'J0934+5514', 'J0938+5428', 'J0940+2935', 'J0942+3547', 'J0944+3424', 'J0944-0038', 'J1016+3754', 'J1024+0524', 'J1025+3622', 'J1044+0353', 'J1105+4444', 'J1112+5503', 'J1113+2930', 'J1119+5130', 'J1129+2034', 'J1132+1411', 'J1132+5722', 'J1144+4012', 'J1148+2546', 'J1150+1501', 'J1157+3220', 'J1200+1343', 'J1225+6109', 'J1253-0312', 'J1314+3452', 'J1323-0132', 'J1359+5726', 'J1414+0540', 'J1416+1223', 'J1418+2102', 'J1428+1653', 'J1429+0643', 'J1444+4237', 'J1448-0110', 'J1521+0759', 'J1525+0757', 'J1545+0858', 'J1612+0817', 'J2103-0728']
N1 = [7.3054524064088655, 7.089361019943413, 7.126576193211004, None, 6.7282482396975345, 7.128641552953883, None, None, 7.842747118699164, 6.925147007593108, 7.445910747438067, None, 7.120458762096645, 6.729553536084745, 6.8169369398277855, None, 6.5250577708577415, 7.025141949625193, 6.